In [1]:

import pandas as pd
import numpy as np

# 1. Load CarDekho dataset directly from URL
url = "https://raw.githubusercontent.com/pujarisurakshashankar/car-market-trend-analysis/main/data/CarDekho_Dataset.csv"
df = pd.read_csv(url)

print("First five records:")
print(df.head())
print("\nColumns:", df.columns.tolist())



data = df[["Selling_Price", "Present_Price",
           "Kms_Driven", "Fuel_Type"]].copy()


data = data.dropna()


data = pd.get_dummies(data, columns=["Fuel_Type"], drop_first=True,
                      dtype=int)

# 2. Separate target and predictors
y = data["Selling_Price"].to_numpy(dtype=float)

feature_columns = ["Present_Price", "Kms_Driven"] + [
    col for col in data.columns if col.startswith("Fuel_Type_")
]
X = data[feature_columns].to_numpy(dtype=float)


np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

train_idx = indices[:split]
test_idx = indices[split:]

X_train_raw = X[train_idx].copy()
X_test_raw = X[test_idx].copy()
y_train = y[train_idx]
y_test = y[test_idx]


mean = X_train_raw[:, :2].mean(axis=0)
std = X_train_raw[:, :2].std(axis=0)
std[std == 0] = 1

X_train_raw[:, :2] = (X_train_raw[:, :2] - mean) / std
X_test_raw[:, :2] = (X_test_raw[:, :2] - mean) / std

# 3. Add bias column and calculate weights using NumPy
X_train = np.column_stack(
    (np.ones(len(X_train_raw)), X_train_raw)
)
X_test = np.column_stack(
    (np.ones(len(X_test_raw)), X_test_raw)
)


w = np.linalg.pinv(X_train.T @ X_train) @ X_train.T @ y_train

print("\nRegression weights:")
print(w)

# 4. Predict selling prices
y_pred = X_test @ w


rss = np.sum((y_test - y_pred) ** 2)


ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
r2 = 1 - (rss / ss_tot)

print("\nActual Selling Prices:", y_test[:10])
print("Predicted Selling Prices:", np.round(y_pred[:10], 2))
print("Residual Sum of Squares (RSS):", round(rss, 4))
print("R-squared Score:", round(r2, 4))


First five records:
  Car_Name  Year  Selling_Price  Present_Price  Kms_Driven Fuel_Type  \
0     ritz  2014           3.35           5.59       27000    Petrol   
1      sx4  2013           4.75           9.54       43000    Diesel   
2     ciaz  2017           7.25           9.85        6900    Petrol   
3  wagon r  2011           2.85           4.15        5200    Petrol   
4    swift  2014           4.60           6.87       42450    Diesel   

  Seller_Type Transmission  Owner  
0      Dealer       Manual      0  
1      Dealer       Manual      0  
2      Dealer       Manual      0  
3      Dealer       Manual      0  
4      Dealer       Manual      0  

Columns: ['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']

Regression weights:
[ 3.82497099  4.27787225 -0.87027198  2.9847855   0.31883765]

Actual Selling Prices: [4.   2.55 5.3  3.25 0.45 5.5  3.15 0.6  0.2  3.5 ]
Predicted Selling Prices: [3.75 2.18 4.1